In [ ]:
!pip install -q peft
!pip install -q "numpy<2.0"
!pip install -q terratorch
# 세션다시 시작

In [ ]:
import os

def count_chip_data_pairs(root_path):
    print(f"🔍 데이터 스캔 시작: {root_path}")

    if not os.path.exists(root_path):
        print("❌ 경로가 존재하지 않습니다. 마운트 상태를 확인하세요.")
        return

    # 영상 ID별 폴더 리스트
    video_folders = [d for d in os.listdir(root_path) if os.path.isdir(os.path.join(root_path, d))]

    total_pairs = 0
    all_pairs = []

    print("\n📊 폴더별 칩(Chip) 매칭 결과")
    print("-" * 65)

    for folder in sorted(video_folders):
        folder_path = os.path.join(root_path, folder)
        files = os.listdir(folder_path)

        # 1. 해당 폴더에서 _sim_s2.tif로 끝나는 입력 파일만 추출
        sim_files = [f for f in files if f.lower().endswith('_sim_s2.tif')]

        match_count = 0
        for sim_file in sim_files:
            # 2. '_sim_s2' 부분만 제거하고 '.tif'를 붙여서 정답 파일명 생성
            # 예: K3A_..._R_0002_K3A_sim_s2.tif -> K3A_..._R_0002_K3A.tif

            # 대소문자 무시하고 '_sim_s2' 앞부분만 가져오기
            ext_index = sim_file.lower().rfind('_sim_s2')
            if ext_index == -1: continue

            gt_file = sim_file[:ext_index] + ".tif"

            # 3. 정답 파일이 실제로 폴더에 존재하는지 확인
            if gt_file in files:
                match_count += 1
                all_pairs.append({
                    'input': os.path.join(folder_path, sim_file),
                    'gt': os.path.join(folder_path, gt_file)
                })

        if match_count > 0:
            display_folder = folder if len(folder) < 45 else f"{folder[:42]}..."
            print(f"📁 {display_folder} : {match_count:3d} chips")
            total_pairs += match_count

    print("-" * 65)
    print(f"✅ 총 학습 가능한 데이터 쌍(Chips): {total_pairs} 개")
    print("=" * 65)

    return all_pairs

# 실행
dataset_path = DATASET_ROOT  # 노트북 상단 설정 셀 참조
valid_data_list = count_chip_data_pairs(dataset_path)

In [ ]:
# ======================================================================
# 경로 설정 — 본인 환경에 맞게 수정하세요
# ======================================================================
import os

# Colab을 쓰는 경우에만 drive mount
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DEFAULT_ROOT = '/content/drive/MyDrive/kompsat_sr'  # ← 본인 드라이브 경로로 수정
except ImportError:
    DEFAULT_ROOT = './data'  # 로컬 환경

PROJECT_ROOT     = os.environ.get('PROJECT_ROOT', DEFAULT_ROOT)
DATASET_ROOT     = os.path.join(PROJECT_ROOT, 'dataset')
CHECKPOINT_DIR   = os.path.join(PROJECT_ROOT, 'checkpoints')
PRITHVI_DIR      = os.path.join(PROJECT_ROOT, 'prithvi')
SR_ROOT          = PROJECT_ROOT
VAL_DATASET_ROOT = os.path.join(PROJECT_ROOT, 'val_dataset')

os.makedirs(PROJECT_ROOT, exist_ok=True)
print(f'✅ PROJECT_ROOT = {PROJECT_ROOT}')


In [ ]:
import random

def split_dataset(data_list, train_ratio=0.8, val_ratio=0.1, seed=42):
    # 결과의 재현성을 위해 시드 고정
    random.seed(seed)
    # 데이터 순서 무작위로 섞기
    random.shuffle(data_list)

    total = len(data_list)
    train_end = int(total * train_ratio)
    val_end = train_end + int(total * val_ratio)

    train_set = data_list[:train_end]
    val_set = data_list[train_end:val_end]
    test_set = data_list[val_end:]

    print(f"📊 데이터 분할 완료")
    print(f"   - Train      : {len(train_set)} 개 (실시간 증강 적용 예정)")
    print(f"   - Validation : {len(val_set)} 개 (Best Model 갱신 체크용)")
    print(f"   - Test       : {len(test_set)} 개 (최종 PSNR/SSIM 측정용)")

    return train_set, val_set, test_set

# 실행
train_list, val_list, test_list = split_dataset(valid_data_list)
print(f"   - Train      : {train_list[0]}")
print(f"   - Validation : {val_list[0]}")
print(f"   - Test       : {test_list[0]}")

==============================
#학습
==============================

In [ ]:
import os
import gc
import numpy as np
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.amp import autocast, GradScaler
from google.colab import drive
from peft import LoraConfig, get_peft_model
from tqdm import tqdm

import timm
import terratorch.models
from terratorch.registry import BACKBONE_REGISTRY

In [ ]:
# =====================================================================
# 0. 드라이브 마운트 및 기본 경로 설정
# =====================================================================
drive.mount('/content/drive')

# 프로젝트 및 데이터 경로 설정
prithvi_path = f'{PRITHVI_DIR}'
encoder_file = os.path.join(prithvi_path, 'Prithvi_EO_V2_300M_TL.pt')
checkpoint_dir = f'{CHECKPOINT_DIR}'
dataset_rppth = f'{DATASET_ROOT}'


In [ ]:
# =====================================================================
# 1. 데이터 정의
# =====================================================================

# =====================================================================
# 1. 데이터 스캔 및 분할 함수 (물리적 이동 없음)
# =====================================================================
def get_data_split(root_path, train_ratio=0.8, val_ratio=0.1, seed=42):
    random.seed(seed)
    all_pairs = []

    # 영상 ID별 폴더 리스트 스캔
    video_folders = [d for d in os.listdir(root_path) if os.path.isdir(os.path.join(root_path, d))]

    for folder in video_folders:
        folder_path = os.path.join(root_path, folder)
        files = os.listdir(folder_path)
        # _sim_s2.tif 파일 찾기
        sim_files = [f for f in files if f.lower().endswith('_sim_s2.tif')]

        for sim_file in sim_files:
            # 매칭 규칙: _sim_s2만 제거하고 .tif 붙이기
            ext_idx = sim_file.lower().rfind('_sim_s2')
            gt_file = sim_file[:ext_idx] + ".tif"

            if gt_file in files:
                all_pairs.append({
                    'input': os.path.join(folder_path, sim_file),
                    'gt': os.path.join(folder_path, gt_file)
                })

    random.shuffle(all_pairs)
    total = len(all_pairs)
    train_idx = int(total * train_ratio)
    val_idx = train_idx + int(total * val_ratio)

    return all_pairs[:train_idx], all_pairs[train_idx:val_idx], all_pairs[val_idx:]

# =====================================================================
# 2. 실시간 증강이 포함된 Dataset 클래스
# =====================================================================
class KOMPSATSRDataset(Dataset):
    def __init__(self, data_list, transform=False):
        self.data_list = data_list
        self.transform = transform
        # Prithvi Normalize 파라미터
        self.mean = np.array([1087.0, 1342.0, 1433.0, 2734.0, 1958.0, 1363.0], dtype=np.float32).reshape(6, 1, 1)
        self.std = np.array([2248.0, 2179.0, 2178.0, 1850.0, 1242.0, 1049.0], dtype=np.float32).reshape(6, 1, 1)

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        paths = self.data_list[idx]

        # 1. 파일 로드
        with rasterio.open(paths['input']) as src:
            lr_img = src.read().astype(np.float32) # (C, H, W)
        with rasterio.open(paths['gt']) as src:
            gt_img = src.read().astype(np.float32)

        # 2. 실시간 증강
        if self.transform:
            if random.random() > 0.5:
                lr_img = np.flip(lr_img, axis=2).copy()
                gt_img = np.flip(gt_img, axis=2).copy()
            if random.random() > 0.5:
                lr_img = np.flip(lr_img, axis=1).copy()
                gt_img = np.flip(gt_img, axis=1).copy()
            k = random.randint(0, 3)
            if k > 0:
                lr_img = np.rot90(lr_img, k, axes=(1, 2)).copy()
                gt_img = np.rot90(gt_img, k, axes=(1, 2)).copy()

        # 3. 채널 직접 할당 (요청하신 부분)
        c, h, w = lr_img.shape
        input_6ch = np.zeros((6, h, w), dtype=np.float32)

        # lr_img가 4채널이라고 가정하고 할당
        input_6ch[0] = lr_img[0]
        input_6ch[1] = lr_img[1]
        input_6ch[2] = lr_img[2]
        input_6ch[3] = lr_img[3]

        # 4. 🌟 정규화 (변수명을 input_6ch로 통일하여 에러 해결)
        input_6ch = (input_6ch - self.mean) / self.std

        # 이미지 스케일 조정
        lr_norm = lr_img / 10000.0
        gt_norm = gt_img / 10000.0

        return {
            'x_6ch': torch.from_numpy(input_6ch), # 모델 입력용
            'lr_img': torch.from_numpy(lr_norm),  # Skip connection용
            'gt_img': torch.from_numpy(gt_norm)   # 정답용
        }

In [ ]:
# =====================================================================
# 2. 모델 아키텍처 정의 (UNet 병렬 엣지 인코더 버전)
# =====================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from peft import get_peft_model, LoraConfig

# ──────────────────────────────────────────
# 1. Attention 모듈
# ──────────────────────────────────────────

class SpatialAttention(nn.Module):
    """CBAM 스타일 공간 어텐션: 엣지 위치 강조"""
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size//2)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class ChannelAttention(nn.Module):
    """채널 어텐션: 중요한 특징 맵 활성화"""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.conv(self.avg_pool(x))

class HybridAttentionBlock(nn.Module):
    """채널 + 공간 어텐션 잔차 블록"""
    def __init__(self, channels=64):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, groups=channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1)
        )
        self.ca = ChannelAttention(channels)
        self.sa = SpatialAttention()

    def forward(self, x):
        out = self.sa(self.ca(self.main(x)))
        return x + out

# ──────────────────────────────────────────
# 2. UNet 병렬 엣지 인코더 (핵심 추가)
# ──────────────────────────────────────────
# 목적: Prithvi와 완전히 병렬로 LR 이미지의
#       다중 스케일 엣지/경계 feature 추출
#
# 출력 스케일:
#   s1: (B, 64,  H,   W  ) ← 픽셀 레벨 엣지 (선, 미세 텍스처)
#   s2: (B, 128, H/2, W/2) ← 오브젝트 경계
#   s3: (B, 256, H/4, W/4) ← 의미 수준 구조

class UNetEdgeEncoder(nn.Module):
    def __init__(self, in_channels=4, base_ch=64):
        super().__init__()
        # Encoder
        self.enc1 = nn.Sequential(
            nn.Conv2d(in_channels, base_ch, 3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base_ch, base_ch, 3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.down1 = nn.Conv2d(base_ch, base_ch * 2, 3, stride=2, padding=1)

        self.enc2 = nn.Sequential(
            nn.Conv2d(base_ch * 2, base_ch * 2, 3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base_ch * 2, base_ch * 2, 3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.down2 = nn.Conv2d(base_ch * 2, base_ch * 4, 3, stride=2, padding=1)

        self.enc3 = nn.Sequential(
            nn.Conv2d(base_ch * 4, base_ch * 4, 3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base_ch * 4, base_ch * 4, 3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
        )

        # 각 스케일별 공간 어텐션 (엣지 위치 강조)
        self.sa1 = SpatialAttention()
        self.sa2 = SpatialAttention()
        self.sa3 = SpatialAttention()

    def forward(self, lr_img):
        s1 = self.sa1(self.enc1(lr_img))           # (B, 64,  H,   W  )
        s2 = self.sa2(self.enc2(self.down1(s1)))   # (B, 128, H/2, W/2)
        s3 = self.sa3(self.enc3(self.down2(s2)))   # (B, 256, H/4, W/4)
        return s1, s2, s3

# ──────────────────────────────────────────
# 3. UnpatchingLayer (Prithvi 토큰 복원)
# ──────────────────────────────────────────

class UnpatchingLayer(nn.Module):
    """Prithvi 토큰을 단계적으로 업샘플링 (14→56→224)"""
    def __init__(self, in_dim=1024, out_dim=64):
        super().__init__()
        self.up1 = nn.ConvTranspose2d(in_dim, out_dim * 2, kernel_size=4, stride=4)
        self.up2 = nn.ConvTranspose2d(out_dim * 2, out_dim, kernel_size=4, stride=4)
        self.smooth = nn.Conv2d(out_dim, out_dim, kernel_size=3, padding=1)
        self.act = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        x = self.act(self.up1(x))
        x = self.act(self.up2(x))
        return self.act(self.smooth(x))

# ──────────────────────────────────────────
# 4. UNet-Guided SR Decoder
# ──────────────────────────────────────────
# 스킵커넥션으로 UNet 엣지 feature를 단계별 융합:
#   Level 3 (H/4): Prithvi context + s3
#   Level 2 (H/2): upsample       + s2
#   Level 1 (H  ): upsample       + s1
#   → PixelShuffle x4 → HR

class PrithviSRDecoder(nn.Module):
    def __init__(self, prithvi_embed_dim=1024, feature_dim=64, out_channels=4):
        super().__init__()
        self.unpatch = UnpatchingLayer(in_dim=prithvi_embed_dim, out_dim=feature_dim)

        # Level 3: Prithvi(64ch) + s3(256ch) → 128ch
        self.fuse3 = nn.Sequential(
            nn.Conv2d(feature_dim + 256, feature_dim * 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.refine3 = HybridAttentionBlock(channels=feature_dim * 2)

        # Level 2: up(128ch) + s2(128ch) → 64ch
        self.up3to2 = nn.ConvTranspose2d(feature_dim * 2, feature_dim, kernel_size=2, stride=2)
        self.fuse2 = nn.Sequential(
            nn.Conv2d(feature_dim + 128, feature_dim, 1),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.refine2 = HybridAttentionBlock(channels=feature_dim)

        # Level 1: up(64ch) + s1(64ch) → 64ch
        self.up2to1 = nn.ConvTranspose2d(feature_dim, feature_dim, kernel_size=2, stride=2)
        self.fuse1 = nn.Sequential(
            nn.Conv2d(feature_dim + 64, feature_dim, 1),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.refine1 = nn.Sequential(
            *[HybridAttentionBlock(channels=feature_dim) for _ in range(3)]
        )

        # 최종 PixelShuffle x4
        self.up_conv = nn.Conv2d(feature_dim, out_channels * 16, kernel_size=3, padding=1)
        nn.init.constant_(self.up_conv.weight, 0)
        nn.init.constant_(self.up_conv.bias, 0)
        self.pixel_shuffle = nn.PixelShuffle(4)

    def forward(self, prithvi_tokens, lr_img, unet_feats):
        s1, s2, s3 = unet_feats
        B = prithvi_tokens.size(0)

        # Prithvi 토큰 → 공간 feature (B, 64, 224, 224)
        x = prithvi_tokens[:, 1:, :].transpose(1, 2).reshape(B, -1, 14, 14)
        global_feat = self.unpatch(x)  # (B, 64, 224, 224)

        # s3 해상도로 맞추기
        global_feat_s3 = F.interpolate(global_feat, size=s3.shape[2:],
                                        mode='bilinear', align_corners=False)

        # Level 3
        f3 = self.refine3(self.fuse3(torch.cat([global_feat_s3, s3], dim=1)))

        # Level 2
        f2 = self.up3to2(f3)
        if f2.shape[2:] != s2.shape[2:]:
            f2 = F.interpolate(f2, size=s2.shape[2:], mode='bilinear', align_corners=False)
        f2 = self.refine2(self.fuse2(torch.cat([f2, s2], dim=1)))

        # Level 1
        f1 = self.up2to1(f2)
        if f1.shape[2:] != s1.shape[2:]:
            f1 = F.interpolate(f1, size=s1.shape[2:], mode='bilinear', align_corners=False)
        f1 = self.refine1(self.fuse1(torch.cat([f1, s1], dim=1)))

        # 잔차 + Bicubic
        residual = self.pixel_shuffle(self.up_conv(f1))
        bicubic_up = F.interpolate(lr_img, scale_factor=4,
                                   mode='bicubic', align_corners=False)
        return residual + bicubic_up

# ──────────────────────────────────────────
# 5. FullSRModel: Prithvi + UNet 병렬
# ──────────────────────────────────────────

class FullSRModel(nn.Module):
    """
    두 브랜치 병렬 인코딩:
      [Branch 1] Prithvi LoRA  → 위성 분광/문맥 특징
      [Branch 2] UNetEdgeEncoder → 다중 스케일 엣지/경계 특징
    두 feature를 디코더에서 스킵커넥션으로 단계별 융합.
    """
    def __init__(self, encoder, decoder, unet_edge_encoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.unet_edge_encoder = unet_edge_encoder

    def forward(self, x_6ch, lr_img):
        # Branch 1: Prithvi (위성 분광)
        encoder_outputs = self.encoder(x_6ch)
        combined_feat = (encoder_outputs[-1] + encoder_outputs[-4]) * 0.5

        # Branch 2: UNet 엣지 인코더 (LR 이미지 직접 입력)
        unet_feats = self.unet_edge_encoder(lr_img)  # (s1, s2, s3)

        return self.decoder(combined_feat, lr_img, unet_feats)

# ──────────────────────────────────────────
# 6. 초기화 함수
# ──────────────────────────────────────────

def get_prithvi_encoder_with_lora(checkpoint_path, model_name='prithvi_eo_v2_300'):
    print("\n[초기화] Prithvi 백본 로드 및 LoRA 어댑터 장착 중...")
    encoder = BACKBONE_REGISTRY.build(model_name, pretrained=False, num_frames=1)
    ckpt = torch.load(checkpoint_path, map_location='cpu')
    state_dict = ckpt.get('state_dict', ckpt.get('model', ckpt))
    encoder.load_state_dict(state_dict, strict=False)

    lora_config = LoraConfig(
        r=16, lora_alpha=16, target_modules=["qkv", "proj"],
        lora_dropout=0.1, bias="none"
    )
    lora_encoder = get_peft_model(encoder, lora_config)
    lora_encoder.print_trainable_parameters()
    return lora_encoder


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# =====================================================================
# 3. 손실 함수 (L1 + SAM + Edge + SSIM — 4개)
# =====================================================================

# --- [Loss 1] SAM Loss (분광 정보 유지) ---
class SAMLoss(nn.Module):
    def __init__(self, eps=1e-7):
        super().__init__()
        self.eps = eps

    def forward(self, pred, target):
        dot_product  = torch.sum(pred * target, dim=1)
        norm_pred    = torch.norm(pred,   dim=1)
        norm_target  = torch.norm(target, dim=1)
        denominator  = norm_pred * norm_target + self.eps
        cos_sim      = torch.clamp(dot_product / denominator,
                                   -1.0 + self.eps, 1.0 - self.eps)
        loss = torch.mean(torch.acos(cos_sim))
        if torch.isnan(loss):
            return torch.tensor(0.0, device=pred.device, requires_grad=True)
        return loss

# --- [Loss 2] Edge Loss (경계선 복원 — Sobel gradient) ---
class EdgeLoss(nn.Module):
    def __init__(self, channels=4):
        super().__init__()
        filter_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])
        filter_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]])
        self.weight_x = filter_x.view(1, 1, 3, 3).repeat(channels, 1, 1, 1)
        self.weight_y = filter_y.view(1, 1, 3, 3).repeat(channels, 1, 1, 1)

    def forward(self, pred, target):
        wx = self.weight_x.to(pred.device).type_as(pred)
        wy = self.weight_y.to(pred.device).type_as(pred)
        pred_dx   = F.conv2d(pred,   wx, padding=1, groups=pred.size(1))
        pred_dy   = F.conv2d(pred,   wy, padding=1, groups=pred.size(1))
        target_dx = F.conv2d(target, wx, padding=1, groups=target.size(1))
        target_dy = F.conv2d(target, wy, padding=1, groups=target.size(1))
        return F.l1_loss(pred_dx, target_dx) + F.l1_loss(pred_dy, target_dy)

# --- [Loss 3] SSIM Loss (구조적 유사도) ---
class SSIMLoss(nn.Module):
    """
    채널별 SSIM을 계산해 1 - SSIM_mean 을 loss로 반환.
    엣지·텍스처 구조 보존에 L1보다 민감하게 반응.
    """
    def __init__(self, window_size=11, channels=4):
        super().__init__()
        self.window_size = window_size
        self.channels    = channels
        self.register_buffer('window', self._make_gaussian_window(window_size, channels))

    @staticmethod
    def _make_gaussian_window(size, channels):
        sigma = 1.5
        coords = torch.arange(size, dtype=torch.float32) - size // 2
        g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
        g = g / g.sum()
        window_2d = g.unsqueeze(0) * g.unsqueeze(1)          # (size, size)
        window    = window_2d.unsqueeze(0).unsqueeze(0)       # (1, 1, size, size)
        return window.expand(channels, 1, size, size).contiguous()

    def forward(self, pred, target):
        C1, C2 = 0.01 ** 2, 0.03 ** 2
        pad    = self.window_size // 2
        w      = self.window.to(pred.device).type_as(pred)

        mu1    = F.conv2d(pred,   w, padding=pad, groups=self.channels)
        mu2    = F.conv2d(target, w, padding=pad, groups=self.channels)
        mu1_sq = mu1 ** 2;  mu2_sq = mu2 ** 2;  mu1_mu2 = mu1 * mu2

        sigma1_sq = F.conv2d(pred   * pred,   w, padding=pad, groups=self.channels) - mu1_sq
        sigma2_sq = F.conv2d(target * target, w, padding=pad, groups=self.channels) - mu2_sq
        sigma12   = F.conv2d(pred   * target, w, padding=pad, groups=self.channels) - mu1_mu2

        ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) /                    ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
        return 1.0 - ssim_map.mean()

# --- [통합] SatelliteSRLoss (L1 + SAM + Edge + SSIM) ---
class SatelliteSRLoss(nn.Module):
    """
    total = alpha * L1 + beta * SAM + gamma * Edge + delta * SSIM

    Warm-up  (epoch < 5) : alpha=1.0, 나머지 0  → L1만으로 안정 수렴
    Refinement           : alpha=1.0, beta=0.1, gamma=0.2, delta=0.1
    """
    def __init__(self, alpha=1.0, beta=0.1, gamma=0.2, delta=0.1):
        super().__init__()
        self.l1_loss   = nn.L1Loss()
        self.sam_loss  = SAMLoss()
        self.edge_loss = EdgeLoss(channels=4)
        self.ssim_loss = SSIMLoss(channels=4)
        self.alpha = alpha
        self.beta  = beta
        self.gamma = gamma
        self.delta = delta

    def forward(self, pred, target):
        l1   = self.l1_loss(pred, target)
        sam  = self.sam_loss(pred, target)
        edge = self.edge_loss(pred, target)
        ssim = self.ssim_loss(pred, target)
        total = self.alpha * l1 + self.beta * sam + self.gamma * edge + self.delta * ssim
        # (total, l1, sam, edge, ssim) — 학습 루프에서 개별 모니터링
        return total, l1, sam, edge, ssim


In [ ]:
# =====================================================================
# 5. 체크포인트 관리 함수
# =====================================================================
def load_checkpoint(model, optimizer, path):
    if os.path.exists(path):
        checkpoint = torch.load(path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        print(f"🚀 체크포인트 발견! {start_epoch} 에폭부터 학습을 재개합니다.")
        return start_epoch, checkpoint['loss']
    print("🆕 기존 체크포인트가 없습니다. 처음부터 학습을 시작합니다.")
    return 0, float('inf')

def save_checkpoint(model, optimizer, epoch, loss, path):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }, path)

In [ ]:
import json

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dataset_root = f'{DATASET_ROOT}'
    history_path = os.path.join(checkpoint_dir, 'loss_history.json')

    # 1. 데이터 분할 및 로더
    train_list, val_list, test_list = get_data_split(dataset_root)
    train_ds = KOMPSATSRDataset(train_list, transform=True)
    val_ds   = KOMPSATSRDataset(val_list,   transform=False)

    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False)

    # 2. 모델 초기화 (Prithvi + UNet 병렬 인코더)
    encoder_lora      = get_prithvi_encoder_with_lora(encoder_file)
    unet_edge_encoder = UNetEdgeEncoder(in_channels=4, base_ch=64).to(device)
    decoder           = PrithviSRDecoder()
    full_model        = FullSRModel(encoder_lora, decoder, unet_edge_encoder).to(device)

    optimizer = optim.AdamW(full_model.parameters(), lr=1e-5, weight_decay=1e-4)
    scaler    = GradScaler('cuda')

    # 3. 체크포인트 로드
    latest_ckpt_path = os.path.join(checkpoint_dir, 'latest_sr_model.pth')
    start_epoch, _   = load_checkpoint(full_model, optimizer, latest_ckpt_path)

    if os.path.exists(history_path):
        with open(history_path, 'r') as f:
            history = json.load(f)
        print(f"📈 기존 {len(history['train_loss'])} 에포크 기록을 불러왔습니다.")
    else:
        history = {
            'train_loss': [], 'val_loss': [],
            'train_l1':   [], 'val_l1':   [],
            'train_sam':  [], 'val_sam':  [],
            'train_edge': [], 'val_edge': [],
            'train_ssim': [], 'val_ssim': [],
        }

    best_val_loss = min(history['val_loss']) if history['val_loss'] else float('inf')

    for epoch in range(start_epoch, 100):
        # ── Train ──
        full_model.train()
        mode = "Warm-up" if epoch < 5 else "Refinement"
        # Warm-up: L1만 / Refinement: 4개 모두 활성화
        alpha, beta, gamma, delta = (1.0, 0.0, 0.0, 0.0) if mode == "Warm-up"                                     else (1.0, 0.1, 0.2, 0.1)
        criterion = SatelliteSRLoss(alpha=alpha, beta=beta,
                                    gamma=gamma, delta=delta).to(device)

        train_loss = train_l1 = train_sam = train_edge = train_ssim = 0.0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [{mode}]"):
            x_6ch  = batch['x_6ch'].unsqueeze(2).to(device)
            lr_img = batch['lr_img'].to(device)
            gt_img = batch['gt_img'].to(device)

            optimizer.zero_grad(set_to_none=True)
            with autocast('cuda'):
                output = full_model(x_6ch, lr_img)
                loss, l1, sam, edge, ssim = criterion(output, gt_img)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            train_l1   += l1.item()
            train_sam  += sam.item()
            train_edge += edge.item()
            train_ssim += ssim.item()

        # ── Validation ──
        full_model.eval()
        val_loss = val_l1 = val_sam = val_edge = val_ssim = 0.0

        with torch.no_grad():
            for batch in val_loader:
                x_6ch  = batch['x_6ch'].unsqueeze(2).to(device)
                lr_img = batch['lr_img'].to(device)
                gt_img = batch['gt_img'].to(device)
                with autocast('cuda'):
                    output = full_model(x_6ch, lr_img)
                    loss, l1, sam, edge, ssim = criterion(output, gt_img)
                val_loss += loss.item()
                val_l1   += l1.item()
                val_sam  += sam.item()
                val_edge += edge.item()
                val_ssim += ssim.item()

        n_tr = len(train_loader);  n_vl = len(val_loader)
        avg_train = train_loss / n_tr;  avg_val  = val_loss / n_vl
        avg_tl1   = train_l1   / n_tr;  avg_vl1  = val_l1   / n_vl
        avg_ts    = train_sam  / n_tr;  avg_vs   = val_sam   / n_vl
        avg_te    = train_edge / n_tr;  avg_ve   = val_edge  / n_vl
        avg_tss   = train_ssim / n_tr;  avg_vss  = val_ssim  / n_vl

        # 기록 저장
        history['train_loss'].append(avg_train);  history['val_loss'].append(avg_val)
        history['train_l1'].append(avg_tl1);       history['val_l1'].append(avg_vl1)
        history['train_sam'].append(avg_ts);        history['val_sam'].append(avg_vs)
        history['train_edge'].append(avg_te);       history['val_edge'].append(avg_ve)
        history['train_ssim'].append(avg_tss);      history['val_ssim'].append(avg_vss)

        with open(history_path, 'w') as f:
            json.dump(history, f)

        print(f"Epoch {epoch+1:3d} [{mode:10s}]")
        print(f"  Train → Total:{avg_train:.4f}  L1:{avg_tl1:.4f}  SAM:{avg_ts:.4f}  Edge:{avg_te:.4f}  SSIM:{avg_tss:.4f}")
        print(f"  Val   → Total:{avg_val:.4f}  L1:{avg_vl1:.4f}  SAM:{avg_vs:.4f}  Edge:{avg_ve:.4f}  SSIM:{avg_vss:.4f}")

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            save_checkpoint(full_model, optimizer, epoch, avg_val,
                            os.path.join(checkpoint_dir, 'best_sr_model.pth'))
            print(f"  🌟 Best Model 갱신 (Val Loss: {best_val_loss:.4f})")

        save_checkpoint(full_model, optimizer, epoch, avg_train, latest_ckpt_path)

if __name__ == '__main__':
    main()


손실함수 별로 에포크다 변화랑 파악
에포크당 지표들이 어떻게 변하는지도

In [ ]:
import matplotlib.pyplot as plt
import json
import os

def plot_loss_history(checkpoint_dir):
    history_path = os.path.join(checkpoint_dir, 'loss_history.json')

    if not os.path.exists(history_path):
        print("❌ 저장된 기록 파일이 없습니다.")
        return

    with open(history_path, 'r') as f:
        history = json.load(f)

    train_loss = history['train_loss']
    val_loss = history['val_loss']
    epochs = range(1, len(train_loss) + 1)

    plt.figure(figsize=(10, 6))
    plt.plot(epochs, train_loss, 'b-', label='Training Loss')
    plt.plot(epochs, val_loss, 'r-', label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Warm-up 구간 표시 (선택사항)
    if len(epochs) > 5:
        plt.axvline(x=5, color='gray', linestyle='--', label='Warm-up End')

    plt.show()

# 실행 (checkpoint_dir은 main 함수에서 쓰는 것과 동일해야 함)
plot_loss_history(f'{CHECKPOINT_DIR}')

================================
# 평가
================================

In [ ]:
val_dataset_root = f'{VAL_DATASET_ROOT}'
origin_dataset_root = f'{DATASET_ROOT}'

In [ ]:
import os

def get_data_triplets(val_root_path, origin_root_path, max_samples=None):
    """
    baseline(모델입력): 원본 폴더의 _S2.tif
    gt(정답): 원본 폴더의 _S2를 제거하고 _K3A.tif
    baseline(비교군): 다른 폴더(val)의 _baseline.tif
    """
    all_data = []

    # 스캔의 기준을 origin_root_path로 설정
    if not os.path.exists(origin_root_path):
        print(f"🚨 [에러] 원본 경로를 찾을 수 없습니다: {origin_root_path}")
        return all_data

    video_folders = [d for d in os.listdir(origin_root_path) if os.path.isdir(os.path.join(origin_root_path, d))]

    for folder in video_folders:
        origin_folder_path = os.path.join(origin_root_path, folder)
        val_folder_path = os.path.join(val_root_path, folder)

        if not os.path.exists(val_folder_path):
            continue

        origin_files = os.listdir(origin_folder_path)

        # 1. origin 폴더에서 _S2.tif 파일 찾기
        # 파일명에 대소문자가 섞여 있을 수 있으므로 lower()로 체크하되 정확한 패턴 매칭
        s2_files = [f for f in origin_files if f.lower().endswith('_s2.tif')]

        for s2_file in s2_files:
            # 📍 [Baseline] 모델 입력 (_S2.tif)
            input_path = os.path.join(origin_folder_path, s2_file)

            # 📍 [GT] 정답 파일 조합 (대문자 _S2를 떼고 _K3A.tif를 붙임)
            # rfind는 대소문자를 구분하므로, 원본 파일명에서 '_S2'나 '_s2'가 시작되는 위치를 찾습니다.
            ext_idx = s2_file.lower().rfind('_s2')
            gt_filename = s2_file[:ext_idx] + "_K3A.tif"
            gt_path = os.path.join(origin_folder_path, gt_filename)

            # 📍 [Bicubic] 비교군 파일 조합 (s2_file 원본 이름에서 .tif 떼고 _bicubic.tif)
            base_name = s2_file[:s2_file.lower().rfind('.tif')]
            bicubic_filename = base_name + "_baseline.tif"
            bicubic_path = os.path.join(val_folder_path, bicubic_filename)

            # 🌟 파일 존재 여부 최종 확인
            if os.path.exists(gt_path) and os.path.exists(bicubic_path):
                all_data.append({
                    'input': input_path,
                    'baseline': bicubic_path,
                    'gt': gt_path
                })

                if max_samples is not None and len(all_data) >= max_samples:
                    return all_data

    return all_data

# =====================================================================
# 🚀 실행부
# =====================================================================
val_dataset_root = f'{VAL_DATASET_ROOT}'
origin_dataset_root = f'{DATASET_ROOT}'

# 테스트용으로 5개 추출
test_data = get_data_triplets(val_dataset_root, origin_dataset_root, max_samples=5)

print(f"✅ 총 {len(test_data)}개의 데이터 쌍을 찾았습니다!\n")
if len(test_data) > 0:
    print("📂 첫 번째 샘플 경로 확인:")
    print(f" - Input (Input): {test_data[0]['input']}")
    print(f" - Baseline (Target): {test_data[0]['baseline']}")
    print(f" - Ground Truth: {test_data[0]['gt']}")

In [ ]:
import random

def split_triplets(all_data, train_ratio=0.8, val_ratio=0.1, seed=42):
    """
    찾아낸 데이터 쌍 리스트를 Train, Val, Test 세트로 분할합니다.
    """
    # 1. 재현성을 위해 시드 고정 및 셔플
    random.seed(seed)
    shuffled_data = all_data.copy()
    random.shuffle(shuffled_data)

    # 2. 인덱스 계산
    total = len(shuffled_data)
    train_end = int(total * train_ratio)
    val_end = train_end + int(total * val_ratio)

    # 3. 데이터 슬라이싱
    train_set = shuffled_data[:train_end]
    val_set = shuffled_data[train_end:val_end]
    test_set = shuffled_data[val_end:]

    print(f"📊 데이터 분할 완료 (Seed: {seed})")
    print(f"   - 전체 데이터: {total}개")
    print(f"   - Train 세트: {len(train_set)}개 ({train_ratio*100:.0f}%)")
    print(f"   - Val 세트:   {len(val_set)}개 ({val_ratio*100:.0f}%)")
    print(f"   - Test 세트:  {len(test_set)}개 ({100-(train_ratio+val_ratio)*100:.0f}%)")

    return train_set, val_set, test_set

# =====================================================================
# 🚀 실행부
# =====================================================================
# 1. 먼저 전체 데이터를 다 긁어옵니다 (max_samples 없이)
all_triplets = get_data_triplets(val_dataset_root, origin_dataset_root)

# 2. 비율에 맞춰 분할 (8:1:1)
train_data, val_data, test_data = split_triplets(all_triplets, train_ratio=0.8, val_ratio=0.1, seed=42)

test_data
# 이제 이 분할된 리스트들을 Dataset 클래스에 넣어서 학습/검증에 사용하면 됩니다.

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import os

def read_and_normalize_with_shape(img_path):
    """ TIF 이미지를 읽어서 (시각화용 이미지, 원본 Shape) 두 가지를 반환합니다. """
    with rasterio.open(img_path) as src:
        img = src.read().astype(np.float32)

    # 원본 차원 형태 저장 (Channels, Height, Width)
    original_shape = img.shape

    # (C, H, W) -> (H, W, C) 로 차원 변경
    img = np.transpose(img, (1, 2, 0))

    # 시각화를 위해 앞의 3채널(RGB)만 잘라내기
    if img.shape[-1] > 3:
        img = img[..., :3]

    # Min-Max 정규화 (0~1)
    img_min, img_max = img.min(), img.max()
    if img_max > img_min:
        img = (img - img_min) / (img_max - img_min)

    return img, original_shape

# 🌟 전체 데이터 중 앞의 5개만 선택
num_visualize = 5
target_samples = test_data[:num_visualize]

print(f"전체 {len(test_data)}개 중 앞의 {len(target_samples)}개 샘플 시각화를 시작합니다...\n")

for i, sample in enumerate(target_samples):
    # 키 이름을 'input'으로 통일
    base_name = os.path.basename(sample['input'])

    # 이미지 로드 및 원본 형태(Shape) 반환
    baseline_img, base_shape = read_and_normalize_with_shape(sample['input'])
    bicubic_img, baseline_shape = read_and_normalize_with_shape(sample['baseline'])
    gt_img, gt_shape = read_and_normalize_with_shape(sample['gt'])

    # 1행 3열 시각화
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    fig.suptitle(f"Sample [{i+1}/{len(target_samples)}] : {base_name}", fontsize=14, fontweight='bold')

    axes[0].imshow(baseline_img)
    axes[0].set_title(f'1. Input (Input)\nShape: {base_shape}', fontsize=12)

    axes[1].imshow(bicubic_img)
    axes[1].set_title(f'2. Baseline (Target)\nShape: {baseline_shape}', fontsize=12)

    axes[2].imshow(gt_img)
    axes[2].set_title(f'3. Ground Truth\nShape: {gt_shape}', fontsize=12)

    for ax in axes:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import os
import random
import torch
import numpy as np
import rasterio

# --- [설정 및 추론] ---
full_model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Prithvi 정규화 파라미터
PRITHVI_MEAN = np.array([1087.0, 1342.0, 1433.0, 2734.0, 1958.0, 1363.0], dtype=np.float32).reshape(6, 1, 1)
PRITHVI_STD = np.array([2248.0, 2179.0, 2178.0, 1850.0, 1242.0, 1049.0], dtype=np.float32).reshape(6, 1, 1)

# 결과를 저장할 전역 변수
inference_results = None

if len(test_data) > 0:
    # 1. 랜덤 샘플 선택
    sample = random.choice(test_data)
    print(f"🎬 테스트 파일명: {os.path.basename(sample['input'])}")

    # 2. 데이터 로드
    with rasterio.open(sample['input']) as src: baseline_img = src.read().astype(np.float32)
    with rasterio.open(sample['baseline']) as src: bicubic_img = src.read().astype(np.float32)
    with rasterio.open(sample['gt']) as src: gt_img = src.read().astype(np.float32)

    # 3. 모델 전처리
    c, h, w = baseline_img.shape
    input_6ch = np.zeros((6, h, w), dtype=np.float32)
    for i in range(min(c, 4)): input_6ch[i] = baseline_img[i]

    x_6ch = (input_6ch - PRITHVI_MEAN) / PRITHVI_STD
    input_tensor = torch.from_numpy(x_6ch).unsqueeze(0).to(device)
    lr_tensor = torch.from_numpy(baseline_img / 10000.0).unsqueeze(0).to(device)

    # 4. 모델 추론
    with torch.no_grad():
        output_tensor = full_model(input_tensor, lr_tensor)

    # 5. 변수에 결과 저장 (시각화 셀에서 사용)
    inference_results = {
        'filename': os.path.basename(sample['input']),
        'baseline': baseline_img,
        'bicubic': bicubic_img,
        'model_output': output_tensor.cpu(), # 시각화를 위해 CPU로 이동
        'gt': gt_img
    }
    print("✅ 모델 추론 및 결과 저장 완료!")
else:
    print("🚨 test_data 리스트가 비어있습니다!")

In [ ]:
import matplotlib.pyplot as plt

def get_stats_text(img_data):
    if torch.is_tensor(img_data): img_data = img_data.numpy()
    return f"Min: {img_data.min():.2f} | Max: {img_data.max():.2f} | Mean: {img_data.mean():.2f}"

def to_display(img_data):
    if torch.is_tensor(img_data): img_data = img_data.squeeze().numpy()
    if img_data.ndim == 3:
        img_data = np.transpose(img_data, (1, 2, 0))
        if img_data.shape[-1] > 3: img_data = img_data[..., :3]

    i_min, i_max = img_data.min(), img_data.max()
    return (img_data - i_min) / (i_max - i_min) if i_max > i_min else img_data

if inference_results is not None:
    print(f"🖼️ 시각화 중인 파일: {inference_results['filename']}")

    imgs = [
        inference_results['baseline'],
        inference_results['bicubic'],
        inference_results['model_output'],
        inference_results['gt']
    ]

    titles = [
        f"1. Input (Baseline) - {imgs[0].shape[1:]}",
        f"2. Bicubic Reference - {imgs[1].shape[1:]}",
        f"3. SR Model Result - {imgs[2].shape[2:]}",
        f"4. Ground Truth - {imgs[3].shape[1:]}"
    ]
    colors = ['gray', 'gray', 'red', 'blue']

    fig, axes = plt.subplots(4, 1, figsize=(12, 40))

    for i, (ax, img, title, color) in enumerate(zip(axes, imgs, titles, colors)):
        ax.imshow(to_display(img))
        ax.set_title(title, fontsize=20, color=color, pad=15, fontweight='bold')
        ax.set_xlabel(get_stats_text(img), fontsize=14, color='darkgreen')
        ax.axis('on')
        ax.set_xticks([]); ax.set_yticks([])

        if i >= 2: # 모델 결과와 GT 테두리 강조
            for spine in ax.spines.values():
                spine.set_edgecolor(color)
                spine.set_linewidth(4)

    plt.tight_layout()
    plt.subplots_adjust(hspace=0.2)
    plt.show()
else:
    print("🚨 먼저 '추론 셀'을 실행하여 결과를 생성하세요!")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def get_stats_text(img_data):
    if torch.is_tensor(img_data): img_data = img_data.detach().cpu().numpy()
    return f"Min: {img_data.min():.1f} | Max: {img_data.max():.1f} | Mean: {img_data.mean():.1f}"

def to_display_kompsat(img_data, stretch_percent=2):
    """
    KOMPSAT 특성에 맞춘 백분위 스트레칭 시각화 함수
    """
    if torch.is_tensor(img_data):
        img_data = img_data.squeeze().cpu().numpy()

    # 1. 채널 축 정합 및 RGB 추출 (KOMPSAT: 0:B, 1:G, 2:R)
    if img_data.ndim == 3:
        img_data = np.transpose(img_data, (1, 2, 0))
        if img_data.shape[-1] >= 3:
            rgb_img = img_data[..., [2, 1, 0]].copy() # R-G-B
        else:
            rgb_img = img_data.copy()
    else:
        rgb_img = img_data.copy()

    # 2. 🌟 핵심: Percentile Stretching (색감 보정)
    # 이미 0~1인 모델 출력값과 원본 DN값을 구분하여 처리
    for i in range(rgb_img.shape[-1]):
        channel = rgb_img[..., i]

        # 유효한 하한/상한값 계산 (0.0~1.0 사이 값인 경우와 0~10000 사이 값인 경우 모두 대응)
        low = np.percentile(channel, stretch_percent)
        high = np.percentile(channel, 100 - stretch_percent)

        # 스트레칭 적용
        if high > low:
            channel = (channel - low) / (high - low)

        rgb_img[..., i] = np.clip(channel, 0, 1)

    return rgb_img

if inference_results is not None:
    print(f"🖼️ KOMPSAT 맞춤형 시각화: {inference_results['filename']}")
    print(f"💡 모드: 2% Percentile Stretching (True Color)")

    imgs = [
        inference_results['baseline'],
        inference_results['bicubic'],
        inference_results['model_output'],
        inference_results['gt']
    ]

    titles = [
        f"1. Input (Baseline) - {imgs[0].shape[1:]}",
        f"2. Bicubic Reference - {imgs[1].shape[1:]}",
        f"3. SR Model Result - {imgs[2].shape[2:]}",
        f"4. Ground Truth - {imgs[3].shape[1:]}"
    ]
    text_colors = ['black', 'black', 'red', 'blue']

    fig, axes = plt.subplots(4, 1, figsize=(12, 40))

    for i, (ax, img, title, t_color) in enumerate(zip(axes, imgs, titles, text_colors)):
        # 🌟 KOMPSAT 전용 스트레칭 함수 적용
        ax.imshow(to_display_kompsat(img, stretch_percent=2))

        ax.set_title(title, fontsize=20, color=t_color, pad=15, fontweight='bold')
        ax.set_xlabel(get_stats_text(img), fontsize=14, color='darkgreen')
        ax.axis('on')
        ax.set_xticks([]); ax.set_yticks([])

        if i >= 2:
            for spine in ax.spines.values():
                spine.set_edgecolor(t_color)
                spine.set_linewidth(4)

    plt.tight_layout()
    plt.subplots_adjust(hspace=0.2)
    plt.show()

In [ ]:
import numpy as np
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from scipy.ndimage import sobel

def calculate_sam(target, reference):
    """
    Spectral Angle Mapper (SAM) 계산
    각도(radian) 값이 낮을수록 분광 보존력이 좋음
    """
    # (H, W, C) 데이터 가정
    dot_product = np.sum(target * reference, axis=-1)
    norm_target = np.sqrt(np.sum(target**2, axis=-1))
    norm_ref = np.sqrt(np.sum(reference**2, axis=-1))

    # 0 나누기 방지
    val = dot_product / (norm_target * norm_ref + 1e-8)
    val = np.clip(val, -1.0, 1.0)
    sam_angles = np.arccos(val)

    return np.mean(sam_angles)

def calculate_edge_rmse(target, reference):
    """
    Sobel 필터를 이용한 경계선 복원 오차 계산
    값이 낮을수록 경계선 복원이 정확함
    """
    def get_edge_mag(img):
        # 흑백(평균)으로 변환 후 엣지 검출
        gray = np.mean(img, axis=-1)
        dx = sobel(gray, axis=0)
        dy = sobel(gray, axis=1)
        return np.sqrt(dx**2 + dy**2)

    edge_t = get_edge_mag(target)
    edge_r = get_edge_mag(reference)
    return np.sqrt(np.mean((edge_t - edge_r)**2))

if inference_results is not None:
    # 데이터 준비 (고정 스케일 10000.0 기준)
    def normalize_fixed(img):
        if torch.is_tensor(img): img = img.squeeze().cpu().numpy()
        if img.ndim == 3 and img.shape[0] < img.shape[2]: img = np.transpose(img, (1, 2, 0))
        if img.max() > 2.0: return np.clip(img / 10000.0, 0, 1)
        return np.clip(img, 0, 1)

    gt_f = normalize_fixed(inference_results['gt'])
    bicubic_f = normalize_fixed(inference_results['bicubic'])
    model_f = normalize_fixed(inference_results['model_output'])

    # 1. 기본 지표
    b_psnr = psnr(gt_f, bicubic_f, data_range=1.0)
    m_psnr = psnr(gt_f, model_f, data_range=1.0)

    b_ssim = ssim(gt_f, bicubic_f, data_range=1.0, channel_axis=-1)
    m_ssim = ssim(gt_f, model_f, data_range=1.0, channel_axis=-1)

    # 2. SAM (분광 정보)
    b_sam = calculate_sam(bicubic_f, gt_f)
    m_sam = calculate_sam(model_f, gt_f)

    # 3. Edge RMSE (경계선 오차)
    b_edge = calculate_edge_rmse(bicubic_f, gt_f)
    m_edge = calculate_edge_rmse(model_f, gt_f)

    # 결과 출력
    print(f"📊 [심층 성능 분석 보고서] : {inference_results['filename']}")
    print("-" * 70)
    print(f"{'Metric':<12} | {'Bicubic':<15} | {'Model (Ours)':<15} | {'Improvement'}")
    print("-" * 70)
    print(f"{'PSNR':<12} | {b_psnr:>15.4f} | {m_psnr:>15.4f} | {m_psnr - b_psnr:>+7.4f} dB")
    print(f"{'SSIM':<12} | {b_ssim:>15.4f} | {m_ssim:>15.4f} | {m_ssim - b_ssim:>+7.4f}")
    print(f"{'SAM (rad)':<12} | {b_sam:>15.4f} | {m_sam:>15.4f} | {m_sam - b_sam:>+7.4f} (↓)")
    print(f"{'Edge RMSE':<12} | {b_edge:>15.4f} | {m_edge:>15.4f} | {m_edge - b_edge:>+7.4f} (↓)")
    print("-" * 70)